In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm

In [ ]:
def load_h5_dataset(file_path):
    print(f"Loading {file_path.name}...")
    with h5py.File(file_path, 'r') as f:
        return f['dataset'][:]

# Load MIT Clean Data (CommSignal2)
raw_clean = load_h5_dataset(MIT_CLEAN_FILE)

print(f"Clean Source Shape: {raw_clean.shape}")

In [ ]:
def generate_awgn(length):
    """
    Generates complex Additive White Gaussian Noise (AWGN).
    This simulates 'Barrage Jamming' which covers the entire frequency band.
    """
    # Standard normal distribution for Real and Imag parts
    # (Mean=0, Std=1)
    real_part = np.random.randn(length)
    imag_part = np.random.randn(length)
    
    noise = real_part + 1j * imag_part
    return noise

def mix_signal(clean_signal, noise_signal, target_sinr_db):
    """
    Mixes signals dynamically based on their actual power in the current slice.
    """
    # Measure Actual Power of the specific slices
    # (Power = Mean of the absolute value squared)
    p_clean = np.mean(np.abs(clean_signal)**2)
    p_noise = np.mean(np.abs(noise_signal)**2)
    
    # Safety check
    if p_noise == 0:
        return clean_signal, noise_signal
        
    # Calculate Scaling Factor
    # SINR_linear = P_clean / P_noise_target
    sinr_linear = 10.0 ** (target_sinr_db / 10.0)
    
    # Scale factor ensures the noise hits the exact SINR target
    noise_scale = np.sqrt(p_clean / (p_noise * sinr_linear))
    
    # Mix
    scaled_noise = noise_signal * noise_scale
    mixture = clean_signal + scaled_noise
    
    return mixture

In [ ]:
def generate_barrage_subset(n_samples, clean_source, sinr_options):
    """
    Generates dataset using Synthetic Gaussian Noise.
    """
    X_data = [] 
    Y_data = [] 
    metadata = []
    
    n_clean_frames = clean_source.shape[0]
    
    print(f"Generating {n_samples} Barrage samples...")
    
    for i in tqdm(range(n_samples)):
        # Random Parameters
        clean_idx = np.random.randint(0, n_clean_frames)
        sinr = np.random.choice(sinr_options)
        
        # Slice Clean Signal
        # Pick random start point for variety
        clean_max_start = clean_source.shape[1] - MIT_SAMPLE_LENGTH
        c_start = np.random.randint(0, clean_max_start + 1) if clean_max_start > 0 else 0
        clean_slice = clean_source[clean_idx, c_start : c_start + MIT_SAMPLE_LENGTH]
        
        noise_slice = generate_awgn(MIT_SAMPLE_LENGTH)
        
        mixture = mix_signal(clean_slice, noise_slice, sinr)
        
        # Store
        X_data.append(mixture)
        Y_data.append(clean_slice)
        
        # Metadata
        # We use noise_frame_id = -1 to indicate "Synthetic"
        meta_entry = {
            "global_index": i,
            "sinr_db": sinr,
            "clean_frame_id": clean_idx,
            "noise_frame_id": -1, 
            "interference_type": "Barrage", 
            "source": "AWGN_Generator",
        }
        metadata.append(meta_entry)
        
    return np.array(X_data), np.array(Y_data), pd.DataFrame(metadata)

# --- EXECUTE ---
X_barrage, Y_barrage, meta_barrage = generate_barrage_subset(MIT_DATASET_SIZE, raw_clean, SINR_LEVELS)

print(f"Generated Barrage Set: {X_barrage.shape}")

In [ ]:
def save_subset(X, Y, df, name): 
    # Save
    np.save(MIT_BARRAGE_X, X.astype(np.complex64))
    np.save(MIT_BARRAGE_Y, Y.astype(np.complex64))
    df.to_csv(MIT_BARRAGE_DATASET_METADATA_FILE, index=False)
    
    print(f"Saved {name} to {MIT_GENERATED_DATA_DIR}")

save_subset(X_barrage, Y_barrage, meta_barrage, "barrage_jamming")